In [1]:
""" 
Import packages
"""
import fitz #pymupdf
import pymupdf4llm #to read to markdown, etc
from pathlib import Path #for paths
#import pathlib
#import os
import math #to check if something is na
import numpy as np
import re #regular expressions
import spacy #nlp stuff
import pandas as pd #data frames

In [ ]:
PdfPath_PreHoc = Path('/Users/ritwikavps/Desktop/GoogleDriveFiles/research/DARCLEPaper2025/PapersForExtraction/') #path for pdf files for pre-hoc extraction
PdfPath_PostHoc1 = Path('/Users/ritwikavps/Desktop/GoogleDriveFiles/research/DARCLEPaper2025/PapersForPosthocExtraction/') #path for pdf files for post-hoc extraction 1
PdfPath_PostHoc2 = Path('/Users/ritwikavps/Desktop/GoogleDriveFiles/research/DARCLEPaper2025/PapersForPosthocExtraction_Rd2/') #path for pdf files for post-hoc extraction 2

#There is an argument that it would be more efficient to use list(ParentPath.glob('PapersFor*')) [Here, the glob searches for the sub-string and returns an iterator, much like for glob in the for loop below,
#and list() converts that to a list]. However, because of the how the data (folders with .pdf files as well as the spreadsheets from covidence and the extraction spreadsheets with paper types
# (e.g., observational, experimental, etc.)) are organised, I want to have more control over the process, and there are only three folders, so this way of doing this is not too much of an issue.
# I do intend to adapt the code for a 'clean' version of the data organisation that is a bit more generally adaptable.  

PdfPathList = [PdfPath_PreHoc,PdfPath_PostHoc1,PdfPath_PostHoc2] #Make list with paths to each of the folders with files for extraction
PdfFilesList = [] #Initialise list to store paths to pdf files to read

for i in PdfPathList: #iterate over path list
    FilesListIter_i = i.glob('*.pdf') #grab pdf file paths in each folder. 
    PdfFilesList.extend(list(FilesListIter_i)) #glob returns an iterator so we want to convert that to a list so we can repeatedly loop over it as needed, ergo the list().
    #extend() adds the list to PdfFilesList such that PdfFilesList remains a flat list (i.e., [filepath_prehoc1, filepath_prehoc2, ... filepath_posthoc1_1, ... filepath_posthoc2_1, ...]) 
    #vs. [[filepath_prehoc1, filepath_prehoc2, ...], [filepath_posthoc1_1, ...], [filepath_posthoc2_1, ...]], which is what append() will do

#print(PdfFilesList) 
print(len(PdfFilesList)) #if done right, this should be the same as the sum of the number of pdf files in each of the PdfPath_<> paths

DoiPatt = r'\bdoi[^a-zA-Z]*'

DoiDf = pd.DataFrame(columns=['DoiTxt','FileName']) #corresponding df


256


In [49]:
Ctr = 0
FileName = []
DoiTxt = []

for i_file, file in enumerate(PdfFilesList): #go through files + get an index if we need it
    with fitz.open(file) as doc: #open current doc(with closes it after with ends)
        FileName.append(file.stem) #gets the file name (i.e., pdf name as saved)

        # #initialise temporary dict to keep track of cases where extraction based on methods and results section has not happened
        # Temp_Dict_NoMethResExtract = {"PaperTitle_ToC": PaperTitle_ToC,
        #                               "PaperTitle_NoToC": PaperTitle_NoToC,
        #                               "FileName": FileName} 

        #--2. Get full text markdown + nlp processing + getting other sections + extraction (by id'ing Methods and results sections, without using TOC) ----------------------------
        Pg1Txt = pymupdf4llm.to_markdown(doc, write_images=False, ignore_graphics=True, pages=[0])

        DoiFound = False

        lines = Pg1Txt.splitlines()
        for line in lines:
            if re.search(DoiPatt, line.strip(), re.IGNORECASE): #regexp search
                Ctr = Ctr + 1
                DoiFound = True
                DoiTxt.append(line)
                #print(line)
                break

        if DoiFound == False:
            DoiTxt.append('')
            print(FileName[i_file])
print(Ctr)
        #pymupdf4llm.to_markdown(doc, write_images=False, ignore_graphics=True) #get full text as markdown

        # if i_file == 1:
        #     break

        

26134546_7482303720005152
Saetre-Turner 2015
Wang 2022
SeungheeHa 2024
project_aspire__spoken_language_intervention.30
26134156_7482200100005152
Warlaumont 2014
Xu 2012
TheRoleOfSiblingsOnInfantLanguageExposureInDaylongAudioRecordings_Galindo
Ritwika 2025
Righter 2025
VanDam 2015
Montag 2020
Yapanel 2009
Ambroseetal14
Warren 2010
Wagner 1985
Charronetal16
PIIS096098222030419X
Roy_New Horizons
qt1092j779
235


In [ ]:
DictForDf = {'FileName':FileName, 
             'DoiTxt': DoiTxt}
OpDf = pd.DataFrame(DictForDf)
OpDf.to_excel('Sample.xlsx',index=False)